In [1]:
import pandas as pd

results_path = {
    'nb'        : 'outputs/ggs/results/NegativeBinomialPiecewise_13f6c147_20260507_2337.csv',
    'dt'        : 'outputs/ggs/results/DecisionTreeModel_c30866d4_20260509_0941.csv',
    'rf'        : 'outputs/ggs/results/RandomForestModel_467bdc30_20260509_1125.csv',
    'xgb'       : 'outputs/ggs/results/XGBModel_43704765_20260511_1444.csv',
    'svr'       : 'outputs/ggs/results/SVRModel_c5ba7e02_20260508_1026.csv',
    'cox'       : 'outputs/ggs/results/CoxPHModel_59d4a8b5_20260512_1106.csv',
    'aft'       : 'outputs/ggs/results/WeibullAFTModel_dd4bcf5e_20260512_1116.csv',
    'frailty'   : 'outputs/ggs/results/CoxFrailty_3da8809d_20260512_2005.csv'
}

cols_params = {
    'nb'        : [
    'feature_set', 'window_size', 'n_components', 'clipping_threshold',
    'alpha', 'alpha_reg', 'l1_ratio', 'mean_S_score', 
    'mean_MAE', 'mean_RMSE', 'mean_C_index',
],
    'dt'        : ['feature_set', 'window_size', 'n_components', 
                'clipping_threshold', 'max_depth', 'min_samples_leaf', 
                'min_samples_split', 'max_features', 'mean_S_score', 
                'mean_C_index', 'mean_MAE', 'mean_RMSE']
,
    'rf'        : ['feature_set', 'window_size', 'n_components', 'clipping_threshold',
               'n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 
               'mean_S_score', 'mean_C_index', 'mean_MAE', 'mean_RMSE'],
    'xgb'       : ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'n_estimators', 'learning_rate', 'max_depth', 'subsample', 'colsample_bytree', 
               'reg_lambda', 'min_child_weight', 'mean_S_score', 'mean_C_index', 'mean_MAE', 
               'mean_RMSE'],
    'svr'       : ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'kernel', 'C', 'epsilon', 'gamma', 'degree', 'mean_S_score', 
               'mean_C_index', 'mean_MAE', 'mean_RMSE']
,
    'cox'       : ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'baseline_estimation_method', 'n_baseline_knots', 'penalizer', 
               'l1_ratio', 'confidence_threshold', 'mean_S_score', 'mean_C_index', 
               'mean_MAE', 'mean_RMSE']
,
    'aft'       : ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'confidence_threshold', 'penalizer', 'l1_ratio', 'fit_intercept', 
               'mean_S_score', 'mean_C_index', 'mean_MAE', 'mean_RMSE'],
    'frailty'   : ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'distribution', 'method', 'tdf', 'confidence_threshold', 'mean_S_score', 
               'mean_C_index', 'mean_MAE', 'mean_RMSE']
}

def read_results(model_result: str) -> pd.DataFrame:
    
    if not isinstance(model_result, str):
        raise ValueError("[Error]: model_result must be a string")
    
    if not model_result in results_path.keys():
        raise ValueError("[Error]: model_result must be a valid model code")
    
    return pd.read_csv(results_path[model_result])

def get_results_resume(model_result: str) -> None:
    
    df_results = read_results(model_result)
    
    total = len(df_results)
    exitosas = df_results['mean_S_score'].notna().sum()
    fallidas = df_results['mean_S_score'].isna().sum()
    n_duplicados = df_results.duplicated(subset=cols_params[model_result]).sum()

    print(f"Total configuraciones: {total}")
    print(f"Exitosas:              {exitosas}")
    print(f"Fallidas (NaN):        {fallidas}")
    print(f"Duplicados:            {n_duplicados}")
    print(f"Únicas:                {total - n_duplicados}") 
    
def view_top_results(model_result: str) -> None:
    
    df_results = read_results(model_result)
    
    df_top = (
        df_results
        .dropna(subset=['mean_S_score'])
        .sort_values('mean_S_score', ascending=True)
        .head(10)
        .reset_index(drop=True)
    )

    display(df_top[cols_params[model_result]])   

# Negative Binomial GLM
El **Negative Binomial GLM** es un modelo de regresión estadística de la familia
de modelos lineales generalizados (GLM), diseñado para datos de conteo con
sobredispersión respecto a la distribución de Poisson. A diferencia de los modelos
de ML clásico (árboles, SVR, XGBoost), no aprende patrones no lineales mediante
particiones del espacio de features — estima directamente los parámetros de una
distribución probabilística para la variable respuesta.

**¿Por qué NB para RUL?**
El RUL en ciclos es una variable de conteo no negativa con variabilidad creciente
hacia el final de la vida del motor — condiciones naturales para un GLM Negative
Binomial. Además, es el único modelo clásico del proyecto que produce intervalos
de confianza nativos vía MLE, lo que lo convierte en el modelo de referencia
probabilístico del pipeline.

**Abordaje en este proyecto**
El modelo recibe directamente la salida del pipeline Nodos 2-4 (ventanas deslizantes
→ features estadísticas → RobustScaler + PCA) como matriz de covariables X.
El target es el RUL clippeado a `clipping_threshold`. Se usa link function `log`
para garantizar predicciones estrictamente positivas.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa si features de memoria/frecuencia aportan sobre el set base |
| `window_size` | 15, 20, 25, 30 | Controla cuántos ciclos históricos captura cada ventana |
| `n_components` | 10, 15, 20 | Trade-off entre retención de varianza y dimensionalidad del GLM |
| `clipping_threshold` | 115, 120, 125, 130 | Define el techo del espacio de predicción piecewise |
| `alpha` | 0.1, 0.5, 1.0, 1.5 | Parámetro de sobredispersión de la NB respecto a Poisson |
| `alpha_reg` | 0.0, 0.1, 0.5 | Fuerza de regularización elastic net sobre β |
| `l1_ratio` | 0.0, 0.5, 1.0 | Mixing L1/L2 — solo activo cuando `alpha_reg > 0` |
| `link_type` | log | Fijo — garantiza predicciones positivas y es el canónico para conteos |

##  Resultados GGS y selección de hiperparámetros

In [2]:
get_results_resume(model_result = 'nb')

Total configuraciones: 6912
Exitosas:              6912
Fallidas (NaN):        0
Duplicados:            0
Únicas:                6912


**Resumen del GGS**
- Configuraciones evaluadas: 6,912 — tasa de éxito: 100%
- Folds: 5 (GroupKFold por motor)

In [3]:
view_top_results(model_result = 'nb')

,feature_set,window_size,n_components,clipping_threshold,alpha,alpha_reg,l1_ratio,mean_S_score,mean_MAE,mean_RMSE,mean_C_index
0,B,30,20,115,0.1,0.0,1.0,2.447365,10.386102,13.364027,0.907819
1,B,30,20,115,0.1,0.0,0.0,2.447365,10.386102,13.364027,0.907819
2,B,30,20,115,0.1,0.0,0.5,2.447365,10.386102,13.364027,0.907819
3,B,30,15,115,0.1,0.0,0.5,2.454957,10.403037,13.387680,0.907667
4,B,30,15,115,0.1,0.0,1.0,2.454957,10.403037,13.387680,0.907667
5,B,30,15,115,0.1,0.0,0.0,2.454957,10.403037,13.387680,0.907667
6,B,30,10,115,0.1,0.0,0.5,2.464231,10.415936,13.415371,0.907645
7,B,30,10,115,0.1,0.0,0.0,2.464231,10.415936,13.415371,0.907645
8,B,30,10,115,0.1,0.0,1.0,2.464231,10.415936,13.415371,0.907645
9,D,30,10,115,0.1,0.0,0.5,2.465530,10.514475,13.418269,0.903689


### Conclusiones

**Pipeline**
- `window_size=30` domina todo el top 10 — ventanas más largas capturan mejor
  la tendencia de degradación acumulada.
- `clipping_threshold=115` es consistente en las 10 mejores configuraciones —
  el espacio predictivo útil no supera los 115 ciclos para este dataset.
- `feature_set=B` (features estadísticas + tendencia) domina sobre C y D.
  Añadir features de memoria o frecuencia no aporta mejora medible — la señal
  de degradación está contenida en la tendencia lineal y distribución local.

**Modelo**
- `alpha=0.1` en todo el top 10 — sobredispersión mínima, cercana a Poisson.
  El RUL clippeado tiene variabilidad moderada y controlada.
- `alpha_reg=0.0` — MLE puro supera a cualquier configuración regularizada.
  El dataset tiene suficiente señal para estimar β sin penalización.
- `l1_ratio` es **irrelevante** cuando `alpha_reg=0.0` — matemáticamente
  correcto, confirmado empíricamente: posiciones 0, 1 y 2 tienen métricas
  idénticas con `l1_ratio` 0.0, 0.5 y 1.0.

**Hiperparámetros seleccionados para producción**

```python
best_params_nb = {
    # Pipeline
    'feature_set':        'B',
    'window_size':        30,
    'n_components':       20,
    'clipping_threshold': 115,
    # Modelo
    'link_type':  'log',
    'alpha':      0.1,
    'alpha_reg':  0.0,
    'l1_ratio':   0.0,
}
```

Se elige `n_components=20` sobre 15 y 10 por la diferencia marginal positiva en
todas las métricas, sin costo adicional en producción. `l1_ratio=0.0` se fija por
convención Ridge — si en el futuro se activa regularización, se parte desde L2 puro.
La selección prioriza **parsimonia dentro de la región óptima**: el modelo más simple
que alcanza el mejor rendimiento medible, con cada decisión respaldada por evidencia
directa del GGS.

# Decision Tree Regressor

El **Decision Tree Regressor** es un modelo de ML clásico que aprende particiones
recursivas axis-aligned del espacio de features, asignando a cada región (hoja)
el promedio del RUL de las muestras de entrenamiento que caen en ella. A diferencia
del NB GLM, no asume ninguna distribución sobre la variable respuesta ni produce
estimaciones probabilísticas — es un estimador puramente no paramétrico y determinista.

**¿Por qué DT para RUL?**
Se incluye como baseline no lineal interpretable y como bloque constructivo del
Random Forest. Permite documentar empíricamente si las particiones axis-aligned
del espacio PCA capturan patrones de degradación competitivos con modelos
probabilísticos, y sirve como referencia de complejidad mínima para el ensemble.

**Abordaje en este proyecto**
Igual que NB, recibe la salida del pipeline Nodos 2-4 como matriz X. El target
es el RUL clippeado a `clipping_threshold`. Las predicciones son el promedio de
RUL de la hoja correspondiente, clippeadas a `[0, clipping_threshold]`.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa si features adicionales mejoran las particiones del árbol |
| `window_size` | 15, 20, 25, 30 | Controla el horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de partición |
| `clipping_threshold` | 115, 120, 125, 130 | Techo del espacio predictivo piecewise |
| `max_depth` | 5, 10, None | Profundidad máxima — controla la complejidad del árbol |
| `min_samples_leaf` | 1, 5, 10 | Mínimo de muestras por hoja — suaviza predicciones |
| `min_samples_split` | 2, 10 | Mínimo de muestras para realizar un split interno |
| `max_features` | sqrt, 1.0 | Fracción de features consideradas por split |

##  Resultados GGS y selección de hiperparámetros

In [4]:
get_results_resume(model_result = 'dt')

Total configuraciones: 6912
Exitosas:              6912
Fallidas (NaN):        0
Duplicados:            0
Únicas:                6912


**Resumen del GGS**
- Configuraciones evaluadas: 6,912 — tasa de éxito: 100%
- Folds: 5 (GroupKFold por motor)

In [5]:
view_top_results(model_result = 'dt')

,feature_set,window_size,n_components,clipping_threshold,max_depth,min_samples_leaf,min_samples_split,max_features,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,A,30,10,115,10.0,10,2,1.0,2.755420,0.893368,7.786387,12.218421
1,A,30,10,115,10.0,10,10,1.0,2.755420,0.893368,7.786387,12.218421
2,A,30,15,115,10.0,10,10,1.0,2.903364,0.893097,7.875859,12.430473
3,A,30,15,115,10.0,10,2,1.0,2.903364,0.893097,7.875859,12.430473
4,A,30,10,115,10.0,1,10,1.0,2.908272,0.891567,7.814363,12.328285
5,A,30,10,115,10.0,5,10,1.0,2.920258,0.891460,7.826275,12.326989
6,A,30,10,115,10.0,5,2,1.0,2.920258,0.891460,7.826275,12.326989
7,B,30,10,115,10.0,10,2,1.0,2.952978,0.891611,7.737483,12.196543
8,B,30,10,115,10.0,10,10,1.0,2.952978,0.891611,7.737483,12.196543
9,D,30,10,115,5.0,5,10,1.0,2.995387,0.880828,8.622769,12.394301


### Conclusiones

**Pipeline**
- `window_size=30` y `clipping_threshold=115` dominan el top 10 — consistente
  con NB, confirmando que estos valores son óptimos independientemente del modelo.
- `feature_set=A` domina las 7 primeras posiciones — resultado contraintuitivo
  respecto a NB que prefería `B`. Para un árbol de decisión, el set más simple
  (80 features → 10 PCs) produce componentes con señal más concentrada y
  particiones más informativas. Sets más ricos diluyen la varianza entre más
  componentes, perjudicando la calidad de los splits axis-aligned.
- `n_components=10` es suficiente — el árbol no se beneficia de más componentes.

**Modelo**
- `max_depth=10` domina el top 9 — ni muy superficial (underfitting) ni ilimitado
  (overfitting). La profundidad óptima refleja la complejidad real de las
  particiones de degradación en el espacio PCA.
- `min_samples_leaf=10` en el top 4 — hojas con mínimo 10 muestras suavizan
  las predicciones, reduciendo los picos extremos característicos del árbol.
- `max_features=1.0` es exclusivo en todo el top 10 — a diferencia de Random
  Forest donde se restringe para decorrelacionar árboles, un árbol individual
  se beneficia de considerar todas las componentes PCA en cada split.
- `min_samples_split` es irrelevante — posiciones 0 y 1 tienen métricas
  idénticas con valores 2 y 10. Se fija en 2 (mínimo) por parsimonia.

**Comparación con NB**

| Métrica | NB | DT | Ventaja |
|---------|----|----|---------|
| S-Score | 2.447 | 2.755 | NB |
| MAE | 10.39 | 7.79 | **DT** |
| RMSE | 13.36 | 12.22 | **DT** |
| C-Index | 0.908 | 0.893 | NB |

DT supera a NB en MAE y RMSE pero produce trayectorias de RUL más ruidosas,
con picos extremos de sobreestimación visibles en la trayectoria. Esto explica
el peor S-Score a pesar del mejor MAE — el árbol comete errores más pequeños
en promedio pero con mayor varianza y episodios de sobreestimación peligrosa.
**NB sigue siendo el modelo de referencia.**

**Hiperparámetros seleccionados para producción**

```python
best_params_dt = {
    # Pipeline
    'feature_set':        'A',
    'window_size':        30,
    'n_components':       10,
    'clipping_threshold': 115,
    # Modelo
    'max_depth':          10,
    'min_samples_leaf':   10,
    'min_samples_split':  2,
    'max_features':       1.0,
}
```

`feature_set=A` con `n_components=10` refleja que el árbol se beneficia de un
espacio de partición compacto y limpio. `min_samples_leaf=10` es la decisión
más importante del modelo — reduce los picos extremos al exigir representatividad
estadística mínima en cada hoja. `min_samples_split=2` se fija en el mínimo
porque su efecto es redundante cuando `min_samples_leaf=10` ya controla la
granularidad del árbol.

# Random Forest Regressor 

El **Random Forest Regressor** es un método ensemble que construye múltiples
árboles de decisión sobre subconjuntos aleatorios del conjunto de entrenamiento
(bagging) y promedia sus predicciones. A diferencia de un árbol individual,
reduce la varianza de las predicciones sin incrementar el bias — exactamente
la limitación principal observada en DecisionTreeModel.

**¿Por qué RF para RUL?**
Es la extensión natural del Decision Tree evaluado anteriormente. El bagging
sobre ventanas de motores distintos produce árboles que capturan patrones de
degradación complementarios, y su promedio genera trayectorias de RUL más
suaves y robustas que un árbol individual. Es el primer modelo del proyecto
que supera a NB en todas las métricas simultáneamente.

**Abordaje en este proyecto**
Igual que DT y NB, recibe la salida del pipeline Nodos 2-4. `n_jobs=1` está
fijo internamente para evitar conflictos con el paralelismo externo del GGS
(joblib/loky). Las predicciones son el promedio de los `n_estimators` árboles,
clippeadas a `[0, clipping_threshold]`.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa el impacto del set de features sobre el ensemble |
| `window_size` | 15, 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de partición del ensemble |
| `clipping_threshold` | 115, 120, 125, 130 | Techo del espacio predictivo piecewise |
| `n_estimators` | 50, 100, 200 | Número de árboles — controla el trade-off varianza/cómputo |
| `max_depth` | 5, 10, None | Profundidad máxima por árbol — regularización implícita |
| `min_samples_leaf` | 1, 5, 10 | Mínimo de muestras por hoja — suaviza predicciones |
| `max_features` | sqrt, 1.0 | Fracción de features por split — decorrelación de árboles |

##  Resultados GGS y selección de hiperparámetros

In [6]:
get_results_resume(model_result = 'dt')

Total configuraciones: 6912
Exitosas:              6912
Fallidas (NaN):        0
Duplicados:            0
Únicas:                6912


**Resumen del GGS**
- Configuraciones evaluadas: 10,368 — tasa de éxito: 100%
- Folds: 5 (GroupKFold por motor)

In [7]:
view_top_results(model_result = 'rf')

,feature_set,window_size,n_components,clipping_threshold,n_estimators,max_depth,min_samples_leaf,max_features,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,A,30,20,115,200,NaN,10,1.0,1.883120,0.908693,6.982621,10.621407
1,A,30,20,115,200,10.0,10,1.0,1.888651,0.910075,6.961162,10.583097
2,A,30,20,115,100,10.0,10,1.0,1.891046,0.910163,6.955357,10.586206
3,A,30,20,115,200,NaN,5,1.0,1.891983,0.908016,7.002764,10.654055
4,A,30,20,115,100,NaN,10,1.0,1.893297,0.908422,6.985039,10.636345
5,A,30,20,115,100,NaN,5,1.0,1.898050,0.907910,7.002233,10.667064
6,A,30,20,115,200,10.0,5,1.0,1.898817,0.910293,6.961137,10.597294
7,A,30,15,115,200,10.0,10,1.0,1.900187,0.910782,6.939218,10.612990
8,A,30,15,115,200,NaN,10,1.0,1.901608,0.909682,6.949389,10.651319
9,A,30,20,115,200,NaN,1,1.0,1.902389,0.906617,7.075391,10.712568


### Conclusiones

**Pipeline**
- `window_size=30` y `clipping_threshold=115` dominan el top 10 — tercer modelo
  consecutivo que confirma estos valores como óptimos del pipeline con independencia
  del modelo subyacente. Son parámetros robustos del dataset, no del modelo.
- `feature_set=A` domina todo el top 10 — consistente con DT. El ensemble de
  árboles también prefiere el espacio PCA más compacto (10 PCs desde 80 features).
  La señal de degradación está bien concentrada en las features estadísticas básicas.
- `n_components=20` domina las primeras 9 posiciones — a diferencia de DT que
  prefería 10. RF aprovecha más componentes porque el bagging introduce suficiente
  diversidad entre árboles para explotar dimensiones adicionales sin overfitting.

**Modelo**
- `n_estimators=200` aparece en 7 de las 10 primeras posiciones, pero la diferencia
  con 100 es marginal — 0.008 S-Score entre posiciones 1 y 2.
- `max_depth=None` ocupa la posición 0 con diferencia mínima sobre `max_depth=10`
  (0.006 S-Score). Con bagging suficiente los árboles profundos se promedian y
  su varianza individual queda controlada.
- `min_samples_leaf=10` domina el top — consistente con DT. La regularización
  por tamaño de hoja es beneficiosa independientemente de si hay ensemble o no.
- `max_features=1.0` es exclusivo en todo el top 10 — el bagging provee suficiente
  decorrelación entre árboles sin necesitar restricción de features por split.

**Comparación acumulada**

| Métrica | NB | DT | **RF** |
|---------|----|----|--------|
| S-Score | 2.447 | 2.755 | **1.883** |
| MAE | 10.39 | 7.79 | **6.98** |
| RMSE | 13.36 | 12.22 | **10.62** |
| C-Index | 0.908 | 0.893 | **0.909** |

RF es el nuevo modelo de referencia — primer modelo que supera a NB en las
cuatro métricas simultáneamente. La reducción de varianza por bagging resuelve
el problema de picos extremos observado en DT, produciendo trayectorias de RUL
más suaves y con menor sobreestimación peligrosa.

**Hiperparámetros seleccionados para producción**

```python
best_params_rf = {
    # Pipeline
    'feature_set':        'A',
    'window_size':        30,
    'n_components':       20,
    'clipping_threshold': 115,
    # Modelo
    'n_estimators':       100,
    'max_depth':          10,
    'min_samples_leaf':   10,
    'max_features':       1.0,
}
```

Se prioriza parsimonia dentro de la región óptima. `n_estimators=100` sobre 200
reduce el cómputo de inferencia a la mitad con diferencia de 0.008 en S-Score —
rendimientos decrecientes del bagging evidentes. `max_depth=10` sobre `None`
añade regularización implícita que compensa la reducción de estimadores,
produciendo un ensemble más robusto que `n_estimators=200` + `max_depth=None`
a pesar de métricas de CV similares. La combinación prioriza robustez estructural
sobre optimización marginal de métricas de validación cruzada.

# XGBoost Regressor

El **XGBoost Regressor** es un método de gradient boosting que construye
árboles de decisión de forma secuencial, donde cada árbol corrige los errores
residuales del conjunto anterior. A diferencia del Random Forest (bagging en
paralelo), XGBoost aprende iterativamente — cada ronda de boosting se enfoca
en los ejemplos más difíciles de predecir, reduciendo especialmente los errores
extremos. Incorpora regularización L1/L2 nativa sobre los pesos de las hojas
y construcción eficiente via histogramas (`tree_method='hist'`).

**¿Por qué XGBoost para RUL?**
El boosting secuencial tiene una propiedad particularmente relevante para RUL:
al enfocarse iterativamente en los errores más grandes — que en este contexto
son las sobreestimaciones tardías del RUL — reduce exactamente el tipo de error
que más penaliza el S-Score NASA. Esto lo distingue de RF (reduce varianza
uniformemente) y SVR (minimiza error cuadrático dentro del tubo ε sin
distinción de dirección).

**Abordaje en este proyecto**
Recibe la salida del pipeline Nodos 2-4. `n_jobs=1` está fijo internamente
para evitar conflictos con el paralelismo externo del GGS. `objective=
'reg:squarederror'` y `tree_method='hist'` también son fijos — MSE como
criterio de optimización y construcción eficiente por histogramas.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa el impacto del set de features sobre el boosting |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de boosting |
| `clipping_threshold` | 115, 120, 125 | Techo del espacio predictivo piecewise |
| `n_estimators` | 100, 200, 300 | Número de rondas de boosting |
| `learning_rate` | 0.05, 0.10 | Shrinkage por ronda — controla velocidad de aprendizaje |
| `max_depth` | 3, 5 | Profundidad máxima por árbol base |
| `subsample` | 0.8, 1.0 | Fracción de ventanas por ronda de boosting |
| `colsample_bytree` | 0.8, 1.0 | Fracción de componentes PCA por árbol |
| `reg_lambda` | 0.1, 1.0, 10.0 | Regularización L2 sobre pesos de hojas |
| `min_child_weight` | 1, 5 | Mínimo de peso en nodos hoja |

##  Resultados GGS y selección de hiperparámetros

In [8]:
get_results_resume(model_result = 'xgb')

Total configuraciones: 31104
Exitosas:              31076
Fallidas (NaN):        28
Duplicados:            0
Únicas:                31104


**Resumen del GGS**
- Configuraciones evaluadas: 31,104 — exitosas: 31,076 — fallidas: 28 (0.09%)
- Folds: 5 (GroupKFold por motor)

In [9]:
view_top_results(model_result = 'xgb')

,feature_set,window_size,n_components,clipping_threshold,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,reg_lambda,min_child_weight,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,A,30,10,115,300,0.10,3,1.0,1.0,0.1,5,1.782339,0.912280,7.096352,10.450048
1,A,30,10,115,300,0.10,3,0.8,0.8,10.0,5,1.785607,0.912034,7.112294,10.420613
2,A,30,10,115,300,0.10,3,1.0,1.0,1.0,1,1.789015,0.912452,7.103761,10.461413
3,A,30,10,115,200,0.10,3,0.8,0.8,10.0,5,1.789890,0.912937,7.119702,10.431387
4,A,30,10,115,200,0.10,3,0.8,0.8,0.1,5,1.791486,0.912326,7.127622,10.455702
5,A,30,10,115,300,0.10,3,1.0,1.0,1.0,5,1.792337,0.912006,7.113119,10.470224
6,A,30,10,115,300,0.10,3,0.8,0.8,0.1,5,1.792370,0.911316,7.146037,10.463624
7,A,30,10,115,300,0.10,3,1.0,1.0,10.0,5,1.792403,0.912306,7.091080,10.447014
8,A,30,10,115,300,0.10,3,1.0,1.0,10.0,1,1.792430,0.912445,7.084765,10.444758
9,A,30,15,115,100,0.05,5,0.8,0.8,1.0,1,1.792568,0.913687,7.110839,10.403835


### Conclusiones

**Pipeline**
- `window_size=30` y `clipping_threshold=115` — quinto modelo consecutivo
  que confirma estos valores como parámetros robustos del dataset.
- `feature_set=A` y `n_components=10` — quinto modelo que confirma el espacio
  PCA compacto como óptimo para modelos basados en particiones del espacio
  de features.

**Modelo**
- `learning_rate=0.10` — exclusivo en las primeras 9 posiciones. Con `max_depth=3`
  y suficientes estimadores, lr=0.10 converge a mejor solución que lr=0.05.
- `max_depth=3` — árboles muy superficiales dominan el top. XGBoost con boosting
  secuencial prefiere modelos base débiles que se corrigen iterativamente —
  contrasta con RF donde `max_depth=10` era óptimo.
- `min_child_weight=5` — domina 7 de las 10 primeras posiciones. Nodos con
  peso mínimo de 5 evitan splits sobre regiones con poca representatividad.
- `subsample`, `colsample_bytree`, `reg_lambda` — sin patrón claro. Distribución
  uniforme en exitosas vs fallidas confirma que son irrelevantes para este dataset.
  Se fijan en sus valores más simples y neutros.

**Análisis de configuraciones fallidas**

Las 28 configuraciones fallidas (0.09%) presentan un patrón determinista:
todas comparten `learning_rate=0.05` + `n_estimators=100` + `window_size=25` + `n_components=20`. 
La combinación de learning rate bajo con pocas rondas
de boosting produce capacidad insuficiente para ajustarse al espacio de 20
componentes PCA, generando predicciones degeneradas en algún fold.
Este resultado confirma la superioridad de `learning_rate=0.10` y valida
la exclusión de estas configuraciones del análisis.

**Comparación acumulada**

| Métrica | NB | DT | RF | SVR | **XGB** |
|---------|----|----|-----|-----|---------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | **1.782** |
| MAE | 10.39 | 7.79 | 6.98 | **6.75** | 7.10 |
| RMSE | 13.36 | 12.22 | 10.62 | **10.24** | 10.45 |
| C-Index | 0.908 | 0.893 | 0.909 | **0.915** | 0.912 |

XGBoost lidera en S-Score pero no domina en MAE/RMSE/C-Index. El boosting
secuencial reduce especialmente las sobreestimaciones tardías — el error más
penalizado por el S-Score NASA — sin reducir el error absoluto promedio tanto
como SVR. El ranking definitivo entre XGB y SVR se resolverá con el test set.

**Hiperparámetros seleccionados para producción**

```python
best_params_xgb = {
    # Pipeline
    'feature_set':        'A',
    'window_size':        30,
    'n_components':       10,
    'clipping_threshold': 115,
    # Modelo
    'n_estimators':       200,
    'learning_rate':      0.1,
    'max_depth':          3,
    'subsample':          1.0,
    'colsample_bytree':   1.0,
    'reg_lambda':         1.0,
    'min_child_weight':   5,
}
```

`n_estimators=200` sobre 300 aplica el mismo principio de rendimientos
decrecientes observado en RF — diferencia de 0.007 en S-Score por 50% más
de cómputo. `subsample=1.0` y `colsample_bytree=1.0` producen un modelo
**determinista**: dados el mismo dataset y `random_state=42`, las predicciones
son idénticas en cada ejecución — propiedad deseable en sistemas de
mantenimiento predictivo donde la auditabilidad es crítica. `reg_lambda=1.0`
es el default de XGBoost, calibrado como punto de equilibrio entre bias y
varianza sin evidencia en el GGS para alejarse de él.

# Support Vector Regression

El **Support Vector Regressor (SVR)** es un modelo de ML clásico basado en
la teoría de máquinas de vectores soporte. A diferencia de los métodos de
árboles, SVR no particiona el espacio de features — encuentra el hiperplano
de margen máximo en un espacio de Hilbert de dimensión (potencialmente)
infinita inducido por una función kernel, minimizando el error fuera de una
banda de tolerancia ε alrededor de la predicción.

**¿Por qué SVR para RUL?**
La degradación de motores turbofan en el espacio PCA forma una variedad suave
y continua. SVR con kernel RBF captura esta estructura mediante interpolación
suave, a diferencia de los árboles que la discretizan con particiones
rectangulares. Es el único modelo no probabilístico del proyecto que opera
directamente sobre la geometría del espacio de features sin asumir linealidad
ni estructura de árbol.

**Abordaje en este proyecto**
Recibe la salida del pipeline Nodos 2-4. El kernel RBF calcula similitudes
entre ventanas en el espacio PCA — espacios compactos (pocos componentes)
favorecen la calidad del kernel al evitar la homogeneización de distancias
en alta dimensión. Las predicciones se clipean a `[0, clipping_threshold]`.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa el impacto del set de features sobre el kernel |
| `window_size` | 15, 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio kernel — afecta calidad de distancias |
| `clipping_threshold` | 115, 120, 125, 130 | Techo del espacio predictivo piecewise |
| `kernel` | rbf, linear, poly | Tipo de función kernel — define el espacio de Hilbert |
| `C` | 0.1, 1.0, 10.0 | Regularización — trade-off margen vs error de entrenamiento |
| `epsilon` | 0.01, 0.1 | Ancho de la banda de tolerancia ε-insensible |
| `gamma` | scale, 0.01 | Coeficiente del kernel RBF — controla el radio de influencia |
| `degree` | 2, 3 | Grado del kernel polinomial — irrelevante cuando kernel=rbf |

##  Resultados GGS y selección de hiperparámetros

In [10]:
get_results_resume(model_result = 'svr')

Total configuraciones: 13824
Exitosas:              13824
Fallidas (NaN):        0
Duplicados:            0
Únicas:                13824


**Resumen del GGS**
- Configuraciones evaluadas: 13,824 — tasa de éxito: 100%
- Folds: 5 (GroupKFold por motor)

In [11]:
view_top_results(model_result = 'svr')

,feature_set,window_size,n_components,clipping_threshold,kernel,C,epsilon,gamma,degree,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,A,30,10,115,rbf,10.0,0.10,scale,2,1.824685,0.914618,6.746469,10.242414
1,A,30,10,115,rbf,10.0,0.10,scale,3,1.824685,0.914618,6.746469,10.242414
2,A,30,10,115,rbf,10.0,0.01,scale,3,1.825322,0.914583,6.747740,10.243197
3,A,30,10,115,rbf,10.0,0.01,scale,2,1.825322,0.914583,6.747740,10.243197
4,A,30,10,115,rbf,10.0,0.10,0.01,2,1.828495,0.915369,6.982069,10.351567
5,A,30,10,115,rbf,10.0,0.10,0.01,3,1.828495,0.915369,6.982069,10.351567
6,A,30,10,115,rbf,10.0,0.01,0.01,3,1.830116,0.915366,6.983046,10.353243
7,A,30,10,115,rbf,10.0,0.01,0.01,2,1.830116,0.915366,6.983046,10.353243
8,B,30,10,115,rbf,1.0,0.10,scale,2,1.849759,0.911947,7.240020,10.445129
9,B,30,10,115,rbf,1.0,0.10,scale,3,1.849759,0.911947,7.240020,10.445129


### Conclusiones

**Pipeline**
- `window_size=30` y `clipping_threshold=115` — cuarto modelo consecutivo que
  confirma estos valores. Son parámetros robustos del dataset, independientes
  del modelo subyacente.
- `feature_set=A` domina las primeras 8 posiciones — cuarto modelo que confirma
  el set simple como óptimo. Con `n_components=10`, el kernel RBF opera en un
  espacio compacto donde las distancias entre ventanas son informativas.
- `n_components=10` — a diferencia de RF (que prefería 20), SVR prefiere el
  espacio más compacto. El kernel RBF calcula distancias euclidianas en el espacio
  PCA — en alta dimensión las distancias se homogenizan (maldición de la
  dimensionalidad), degradando la discriminación del kernel.

**Modelo**
- `kernel=rbf` — exclusivo en todo el top 10. El kernel gaussiano captura la
  estructura suave y continua de la degradación en el espacio PCA, superando
  las particiones discretas de los árboles.
- `C=10.0` — domina las primeras 8 posiciones. Regularización baja — el modelo
  prefiere ajustarse bien a los datos con margen de tolerancia reducido.
- `gamma='scale'` — superior a `gamma=0.01` en las primeras 4 posiciones.
  `scale` usa `1/(n_features × var(X))` — se adapta automáticamente a la
  escala de las componentes PCA sin requerir ajuste manual.
- `epsilon` — prácticamente irrelevante. Diferencia de 0.001 en S-Score entre
  0.10 y 0.01. Se fija en 0.1 por convención.
- `degree` — completamente irrelevante con `kernel='rbf'`. Solo aplica al
  kernel polinomial, ausente en el top 10. Se fija en 2 por parsimonia.

**Comparación acumulada**

| Métrica | NB | DT | RF | **SVR** |
|---------|----|----|-----|---------|
| S-Score | 2.447 | 2.755 | 1.883 | **1.825** |
| MAE | 10.39 | 7.79 | 6.98 | **6.75** |
| RMSE | 13.36 | 12.22 | 10.62 | **10.24** |
| C-Index | 0.908 | 0.893 | 0.909 | **0.915** |

SVR supera a RF en las cuatro métricas simultáneamente, convirtiéndose en el
modelo con mejor rendimiento en validación cruzada. La interpolación suave del
kernel RBF se adapta mejor a la geometría continua de la degradación que las
particiones rectangulares del ensemble de árboles. Sin embargo, la diferencia
con RF es marginal (0.058 en S-Score) y el ranking definitivo se determinará
con el conjunto de test.

**Hiperparámetros seleccionados para producción**

```python
best_params_svr = {
    # Pipeline
    'feature_set':        'A',
    'window_size':        30,
    'n_components':       10,
    'clipping_threshold': 115,
    # Modelo
    'kernel':  'rbf',
    'C':       10.0,
    'epsilon': 0.1,
    'gamma':   'scale',
    'degree':  2,
}
```

`kernel='rbf'` con `gamma='scale'` es la combinación canónica para datos
continuos normalizados — el PCA garantiza componentes centradas y con varianza
controlada, condición ideal para `scale`. `C=10.0` refleja que el dataset
tiene suficiente señal para justificar baja regularización. `epsilon=0.1` y
`degree=2` se fijan en sus valores más simples dado que su efecto es nulo o
marginal en este contexto.

# Cox Proportional Hazards 

El **Cox Proportional Hazards (Cox PH)** es un modelo semi-paramétrico de
análisis de supervivencia que estima el riesgo de fallo en función de
covariables sin asumir una forma paramétrica para la baseline hazard h₀(t):

    h(t|X) = h₀(t) · exp(β'X)

A diferencia de los modelos de regresión del proyecto (NB, SVR, DT, RF, XGB),
Cox PH no predice el RUL directamente — estima la función de supervivencia
S(t|X) = P(T > t|X) y deriva el RUL mediante la curva de muerte
F(t|X) = 1 - S(t|X): el primer tiempo t* donde F(t*) = confidence_threshold
define el ciclo de fallo predicho, y RUL = t* - t_stop_actual.

**¿Por qué Cox PH para RUL?**
Cox PH es metodológicamente correcto para datos de degradación con evento
terminal observado — cada motor falla exactamente una vez, lo que se alinea
naturalmente con el marco de supervivencia. Es el modelo de referencia en la
literatura de PHM basada en supervivencia y permite comparar directamente con
los modelos de regresión del proyecto.

**Abordaje en este proyecto**
Se usa `survival::coxph` con estimación de baseline por el método de Breslow
(no paramétrico) o splines cúbicos. La función de supervivencia S(t|X) se
computa via `predict_survival_function()` de lifelines. t_stop de cada ventana
se usa como duración — tiempo absoluto acumulado desde el arranque del motor.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa si features adicionales mejoran la estimación de β |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de covariables |
| `clipping_threshold` | 115, 120, 125 | Techo del RUL predicho |
| `baseline_estimation_method` | breslow, spline | Estimación no paramétrica vs paramétrica de h₀(t) |
| `n_baseline_knots` | 3, 5 | Knots para baseline spline — irrelevante con breslow |
| `penalizer` | 0.0, 0.1, 1.0 | Regularización L2 sobre β |
| `l1_ratio` | 0.0, 0.5, 1.0 | Mixing L1/L2 — solo activo con penalizer > 0 |
| `confidence_threshold` | 0.3, 0.5, 0.95 | Umbral de F(t) para declarar fallo predicho |

##  Resultados GGS y selección de hiperparámetros

In [12]:
get_results_resume(model_result = 'cox')

Total configuraciones: 11664
Exitosas:              2592
Fallidas (NaN):        9072
Duplicados:            0
Únicas:                11664


**Resumen del GGS**
- Configuraciones evaluadas: 11,664
- Exitosas: 2,592 (22.2%) — Fallidas: 9,072 (77.8%)
- Folds: 5 (GroupKFold por motor)

In [13]:
view_top_results(model_result = 'cox')

,feature_set,window_size,n_components,clipping_threshold,baseline_estimation_method,n_baseline_knots,penalizer,l1_ratio,confidence_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,A,20,20,125,breslow,3,0.1,0.0,0.3,7181.622052,0.543478,32.529942,49.373263
1,A,20,20,125,breslow,5,0.1,0.0,0.3,7181.622052,0.543478,32.529942,49.373263
2,A,20,20,120,breslow,3,0.1,0.0,0.3,7181.842354,0.544783,34.543207,49.456894
3,A,20,20,120,breslow,5,0.1,0.0,0.3,7181.842354,0.544783,34.543207,49.456894
4,A,20,20,115,breslow,5,0.1,0.0,0.3,7182.274925,0.546284,36.714807,49.771556
5,A,20,20,115,breslow,3,0.1,0.0,0.3,7182.274925,0.546284,36.714807,49.771556
6,A,20,10,125,breslow,3,0.1,0.0,0.3,7192.910285,0.543419,32.531975,49.377522
7,A,20,10,125,breslow,5,0.1,0.0,0.3,7192.910285,0.543419,32.531975,49.377522
8,A,20,15,125,breslow,5,0.1,0.0,0.3,7192.939122,0.543419,32.532400,49.378036
9,A,20,15,125,breslow,3,0.1,0.0,0.3,7192.939122,0.543419,32.532400,49.378036


### Conclusiones

**El problema de la tasa de eventos**

El pipeline de ventanas deslizantes con paso 1 transforma el dataset así:

```
140 motores × ~170 ciclos promedio = ~23,800 ventanas
Eventos observados: 140 (uno por motor, en el último ciclo)
Tasa de eventos: 140 / 23,800 ≈ 0.59%
```

Cox PH ve 23,800 observaciones de las cuales 23,660 son censuras. La partial
likelihood de Cox no tiene suficiente información para estimar β — la superficie
de likelihood es casi plana y el optimizador converge hacia β ≈ 0 para todas
las componentes PCA.

**Consecuencia en la baseline hazard de Breslow**

Con β ≈ 0, todas las ventanas producen la misma curva de supervivencia:

```
S(t|X) ≈ S₀(t)  para todas las ventanas
```

La baseline de Breslow coloca masa únicamente en los tiempos donde ocurren
eventos — concentrados en los últimos ciclos de cada motor. El resultado es
S(t) = 1.0 durante el 99.4% de la vida del motor, con caída abrupta solo
al final. F(t) raramente supera 0.3 — explicando por qué `confidence_threshold
= 0.3` es el único umbral que produce predicciones no-NaN, y por qué el
77.8% de configuraciones falla completamente.

**Por qué spline falla más que breslow**

La baseline spline requiere estimar adicionalmente los parámetros de los knots.
Con 0.4% evento rate, no hay suficiente información para estimar simultáneamente
β y la forma de la baseline — el optimizador falla en la mayoría de casos.
Breslow produce siempre una estimación válida aunque sea casi plana.

**Este problema no depende de la representación de features**

Un resultado crítico: la incompatibilidad no es una limitación del pipeline
de ventanas deslizantes ni de las features PCA — es una propiedad fundamental
del dataset. Con sensores crudos el problema es exactamente el mismo:

```
140 motores × ~170 ciclos = ~23,800 filas
Tasa de eventos: 140 / 23,800 ≈ 0.59% — idéntica
```

La tasa de eventos depende exclusivamente de la naturaleza del dataset —
un evento por motor por definición en C-MAPSS FD001 — no de cómo se
construyen las features. Esto revela una tensión fundamental e irresoluble:

> *Para explotar la información temporal de los sensores se necesita el
> formato fila-por-ciclo. Pero ese formato produce una tasa de eventos
> estructuralmente baja que hace que Cox PH sea inviable. El formato que
> enriquece la información destruye la señal de supervivencia.*

La única alternativa sería usar una fila por motor — el formato clásico de
supervivencia — pero eso elimina toda la riqueza temporal de los sensores,
que es precisamente lo que hace útil el pipeline de degradación.

**El C-Index como confirmación**

C-Index ≈ 0.544 — prácticamente aleatorio (0.5 = random). El modelo no
discrimina entre motores en distintos estados de degradación. Con β ≈ 0,
todas las predicciones son casi idénticas independientemente del estado real
del motor.

**Comparación acumulada**

| Métrica | NB | DT | RF | SVR | XGB | **CoxPH** |
|---------|----|----|-----|-----|-----|-----------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | 1.782 | **7,182** |
| MAE | 10.39 | 7.79 | 6.98 | 6.75 | 7.10 | **32.5** |
| RMSE | 13.36 | 12.22 | 10.62 | 10.24 | 10.45 | **49.4** |
| C-Index | 0.908 | 0.893 | 0.909 | 0.915 | 0.912 | **0.544** |

Por lo tanto, Cox PH no es competitivo para este dataset y pipeline. La tasa de eventos
de 0.59% hace que el modelo sea estadísticamente inviable — no por limitaciones
de implementación sino por incompatibilidad estructural entre el formato
fila-por-ciclo requerido para explotar los sensores y el formato fila-por-motor
que necesita Cox para estimar β de forma significativa.

**No se seleccionan hiperparámetros de producción.** Cox PH queda documentado
como resultado negativo con valor metodológico: confirma que los modelos de
supervivencia semi-paramétricos clásicos requieren tasas de eventos
significativamente mayores que las producidas por pipelines de ventanas
deslizantes sobre C-MAPSS FD001.

# Weibull Accelerated Failure Time 

El **Weibull Accelerated Failure Time (AFT)** es un modelo paramétrico de
análisis de supervivencia que modela directamente el logaritmo del tiempo
de fallo como función lineal de las covariables:

    log(T) = Xβ + σε,  ε ~ Weibull

A diferencia de Cox PH que modela el hazard ratio relativo h(t|X)/h₀(t),
AFT modela el tiempo de fallo absoluto — las covariables "aceleran" o
"desaceleran" el proceso de degradación multiplicando el tiempo de vida
por exp(-β'X). Esto hace que AFT sea más interpretable que Cox en contextos
de ingeniería: un coeficiente βⱼ negativo significa que la componente PCA j
acelera el fallo.

**¿Por qué Weibull AFT para RUL?**
AFT fue evaluado como alternativa a Cox PH tras documentar la inestabilidad
numérica de los modelos de frailty. La distribución Weibull es el modelo
paramétrico estándar para tiempos de fallo en sistemas mecánicos — su
función de hazard monótonamente creciente (β_forma > 1) es coherente con
la degradación progresiva de motores turbofan. Adicionalmente, AFT con
full likelihood puede capturar información distribucional que Cox PH ignora
al usar solo partial likelihood.

**Abordaje en este proyecto**
Se usa `WeibullAFTFitter` de lifelines. t_stop de cada ventana se usa como
duración. RUL se estima via la curva de muerte F(t|X) = 1 - S(t|X):
el primer t* donde F(t*) = confidence_threshold define el ciclo de fallo
predicho, y RUL = t* - t_stop_actual.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa si features adicionales mejoran la estimación Weibull |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de covariables |
| `clipping_threshold` | 115, 120, 125 | Techo del RUL predicho |
| `confidence_threshold` | 0.3, 0.5, 0.95 | Umbral de F(t) para declarar fallo predicho |
| `penalizer` | 0.0, 0.1, 1.0 | Regularización elastic net sobre β |
| `l1_ratio` | 0.0, 0.5, 1.0 | Mixing L1/L2 — activo cuando penalizer > 0 |
| `fit_intercept` | True, False | Intercepto en el predictor lineal de escala Weibull |

##  Resultados GGS y selección de hiperparámetros

In [14]:
get_results_resume(model_result = 'aft')

Total configuraciones: 5832
Exitosas:              1992
Fallidas (NaN):        3840
Duplicados:            0
Únicas:                5832


**Resumen del GGS**
- Configuraciones evaluadas: 5,832
- Exitosas: 1,992 (34.2%) — Fallidas: 3,840 (65.8%)
- Folds: 5 (GroupKFold por motor)

In [15]:
view_top_results(model_result = 'aft')

,feature_set,window_size,n_components,clipping_threshold,confidence_threshold,penalizer,l1_ratio,fit_intercept,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,D,20,10,115,0.3,1.0,0.5,True,209.778083,0.855188,38.539443,47.167833
1,C,30,15,115,0.3,1.0,1.0,True,224.588502,0.840613,40.949924,48.788670
2,D,20,10,120,0.3,1.0,0.5,True,259.351601,0.851152,40.236892,48.688513
3,C,20,10,115,0.3,1.0,1.0,True,277.323689,0.844367,39.500991,48.619324
4,C,30,15,120,0.3,1.0,1.0,True,277.699692,0.835792,42.909206,50.376978
5,C,25,10,115,0.3,1.0,1.0,True,286.776110,0.837004,40.612826,49.426286
6,B,30,15,115,0.3,1.0,1.0,True,290.787187,0.825507,41.020210,49.670462
7,C,30,20,115,0.3,1.0,1.0,True,290.787187,0.825507,41.020210,49.670462
8,C,25,20,115,0.3,1.0,1.0,True,306.116283,0.831109,40.338076,49.512389
9,C,20,20,115,0.3,1.0,1.0,True,308.907711,0.835247,39.966327,49.388695


### Conclusiones

**Tasa de fallos y causa**

El 65.8% de configuraciones fallidas responde a la misma causa estructural
que CoxPH — tasa de eventos de 0.59% — pero con un agravante adicional:
Weibull AFT usa **full likelihood** en lugar de partial likelihood, lo que
requiere estimar simultáneamente β, la forma ρ y la escala σ de la
distribución. Con tan pocos eventos, el optimizador diverge con más
frecuencia que Cox. La regularización fuerte (`penalizer=1.0`, exclusiva
en el top 10) es la única forma de estabilizar la estimación — a costa de
shrinkear β hacia cero.

**El dato más relevante: C-Index = 0.855**

AFT produce el C-Index más alto de los modelos de supervivencia y
sorprendentemente competitivo con los modelos de regresión (SVR=0.915,
RF=0.909). Esto revela que AFT discrimina el ranking de degradación con
cierta precisión — sabe *quién* está más degradado — pero sus predicciones
absolutas de RUL son pobres (MAE=38.5) porque β≈0 bajo regularización
fuerte colapsa las predicciones hacia la mediana de la distribución Weibull.

La separación entre C-Index competitivo y MAE/RMSE inaceptables es la
firma característica de un modelo cuya capacidad discriminativa es real
pero cuya calibración absoluta está comprometida por la escasez de eventos.

**Diferencias clave respecto a CoxPH**

AFT prefiere `feature_set=C` y `D` — completamente opuesto a los modelos
de regresión que preferían `A`. La full likelihood de Weibull necesita más
información distribucional para estimar la forma y escala — las features
de memoria (C) y frecuencia (D) aportan información sobre la distribución
temporal de la degradación que el set básico `A` no captura. Este es el
único modelo del proyecto donde `feature_set=A` no es óptimo.

`penalizer=1.0` domina en AFT vs `penalizer=0.1` en Cox — la full
likelihood es más inestable que la partial likelihood ante escasez de
eventos y requiere regularización más agresiva para converger.

**La incompatibilidad es estructural e independiente de las features**

Igual que CoxPH, el problema no es la representación de covariables sino
la naturaleza del dataset. Con sensores crudos la tasa de eventos sería
idéntica — un evento por motor por definición en C-MAPSS FD001:

```
n_eventos / n_filas = 140 / 23,800 ≈ 0.59%  — invariante
```

La tensión fundamental entre formato fila-por-ciclo (necesario para
explotar los sensores) y tasa de eventos suficiente (necesaria para
los modelos de supervivencia) es irresoluble para este dataset.

**Comparación acumulada**

| Métrica | NB | DT | RF | SVR | XGB | CoxPH | **AFT** |
|---------|----|----|-----|-----|-----|-------|---------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | 1.782 | 7,182 | **210** |
| MAE | 10.39 | 7.79 | 6.98 | 6.75 | 7.10 | 32.5 | **38.5** |
| RMSE | 13.36 | 12.22 | 10.62 | 10.24 | 10.45 | 49.4 | **47.2** |
| C-Index | 0.908 | 0.893 | 0.909 | 0.915 | 0.912 | 0.544 | **0.855** |

AFT supera a CoxPH en todas las métricas y produce un C-Index
sorprendentemente competitivo (0.855). Sin embargo, las predicciones
absolutas de RUL son inaceptables para uso en producción — MAE de
38.5 ciclos representa un error del ~33% sobre el rango útil de
predicción (0-115 ciclos).

Weibull AFT es el modelo de supervivencia más informativo del proyecto —
su C-Index de 0.855 sugiere que con una tasa de eventos mayor podría
ser un modelo serio. Sin embargo, comparte la incompatibilidad estructural
de CoxPH con el pipeline de ventanas deslizantes: la tasa de eventos de
0.59% hace inviable la estimación estable de los parámetros Weibull sin
regularización agresiva que compromete la calibración absoluta.

**No se seleccionan hiperparámetros de producción.** Weibull AFT queda
documentado como resultado negativo con matiz: a diferencia de CoxPH
(C-Index≈aleatorio), AFT demuestra capacidad discriminativa real que
no puede traducirse en predicciones precisas de RUL por la escasez
estructural de eventos en C-MAPSS FD001.

# Cox con Frailty Compartida

El **Cox con Frailty Compartida (CoxFrailty)** es una extensión del modelo
Cox PH estándar que introduce un término de heterogeneidad aleatoria ω_i
por grupo de observaciones:

    h(t|X, ω_i) = h₀(t) · ω_i · exp(β'X)

donde ω_i sigue una distribución gamma, gaussiana o t compartida entre
todas las ventanas del motor i. El término de frailty captura la
heterogeneidad no observada entre motores — variabilidad en condiciones
de operación, desgaste inicial, o patrones de degradación idiosincrásicos
que las covariables PCA no explican completamente.

**¿Por qué CoxFrailty para RUL?**
El pipeline de ventanas deslizantes genera múltiples filas por motor,
violando el supuesto de independencia del Cox PH estándar. CoxFrailty
es la corrección metodológicamente correcta — el término ω_i agrupa
las ventanas del mismo motor y modela su correlación intra-motor
explícitamente. Es el modelo estadísticamente más riguroso del proyecto
para datos longitudinales con múltiples observaciones por sujeto.

**Abordaje en este proyecto**
Se usa `survival::coxph` con `frailty(motor_id, distribution)` via
la capa `r_repository`. La estimación de θ (varianza de frailty) usa
el algoritmo EM (`method='em'`) o minimización de AIC (`method='aic'`).
`maxit=50` está fijo — valor calibrado empíricamente para viabilidad
computacional en el GGS. RUL se estima via la curva de muerte
F(t|X) = 1 - S(t|X) marginalizando sobre la distribución de frailty.

**Nota computacional**
CoxFrailty es el modelo más costoso del proyecto — ~45 segundos por
configuración con 140 motores y 5 folds. El GGS completo de 3,888
configuraciones requirió ejecución en paralelo (`--jobs 7`).

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa el impacto del set de features sobre la frailty |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de covariables |
| `clipping_threshold` | 115, 120, 125 | Techo del RUL predicho |
| `distribution` | gamma, gaussian, t | Distribución del término de frailty ω_i |
| `method` | em, aic | Método de estimación de la varianza de frailty θ |
| `tdf` | 3, 5 | Grados de libertad de la frailty t — ignorado para gamma/gaussian |
| `confidence_threshold` | 0.3, 0.5, 0.95 | Umbral de F(t) para declarar fallo predicho |

##  Resultados GGS y selección de hiperparámetros

In [16]:
get_results_resume(model_result = 'frailty')

Total configuraciones: 3888
Exitosas:              3888
Fallidas (NaN):        0
Duplicados:            0
Únicas:                3888


**Resumen del GGS**
- Configuraciones evaluadas: 3,888 — exitosas: 3,888 (100%) — fallidas: 0
- Folds: 5 (GroupKFold por motor)

In [18]:
view_top_results(model_result = 'frailty')

,feature_set,window_size,n_components,clipping_threshold,distribution,method,tdf,confidence_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,C,20,20,125,t,em,5,0.3,3390.110036,0.568503,29.384668,45.487590
1,C,20,20,125,t,aic,5,0.3,3390.110036,0.568503,29.384668,45.487590
2,C,20,20,120,t,aic,5,0.3,3390.343479,0.570607,31.405497,45.582263
3,C,20,20,120,t,em,5,0.3,3390.343479,0.570607,31.405497,45.582263
4,C,20,20,115,t,aic,5,0.3,3390.787984,0.573023,33.584662,45.927723
5,C,20,20,115,t,em,5,0.3,3390.787984,0.573023,33.584662,45.927723
6,C,20,20,125,gamma,em,5,0.3,3391.062953,0.568667,29.381488,45.478455
7,C,20,20,125,gamma,em,3,0.3,3391.062953,0.568667,29.381488,45.478455
8,C,20,20,120,gamma,em,5,0.3,3391.296397,0.570776,31.402318,45.573180
9,C,20,20,120,gamma,em,3,0.3,3391.296397,0.570776,31.402318,45.573180


### Conclusiones

**Explicación: 100% de convergencia**

A diferencia de CoxPH (22.2% exitosas) y AFT (34.2% exitosas), CoxFrailty
converge en todas las configuraciones. El término de frailty θ actúa como
regularización implícita sobre la heterogeneidad entre motores, estabilizando
numéricamente la estimación de β incluso con 0.4% evento rate. Esto resuelve
el problema numérico pero no el estadístico — β sigue siendo ≈0 porque la
partial likelihood no tiene suficiente información para estimar coeficientes
significativos con tan pocos eventos.

**C-Index ≈ 0.569 — capacidad discriminativa casi nula**

El C-Index de 0.569 es prácticamente aleatorio (0.5 = random) y muy inferior
al de AFT (0.855). La frailty captura variabilidad entre motores pero no mejora
la discriminación entre ventanas en distintos estados de degradación — β≈0
implica que todas las ventanas producen curvas de supervivencia casi idénticas
independientemente del estado del motor.

**Patrones del top 10**

Pipeline: `feature_set=C`, `window_size=20`, `n_components=20` son exclusivos.
Igual que AFT, CoxFrailty necesita features más ricas (memoria + tendencia)
para alimentar el término de frailty — `feature_set=A` no aporta suficiente
variabilidad entre ventanas para estimar θ.

Modelo: `distribution=t` ocupa las primeras 6 posiciones — colas pesadas más
robustas ante heterogeneidad extrema con pocos eventos. `method` es
completamente irrelevante — posiciones 0 y 1 tienen métricas idénticas con
`em` y `aic`, confirmando que el método de estimación de θ no importa cuando
θ no es identificable estadísticamente. `confidence_threshold=0.3` es exclusivo
— mismo patrón que CoxPH y AFT.

**La incompatibilidad estructural persiste**

El término de frailty no resuelve el problema fundamental documentado para
CoxPH y AFT — la tasa de eventos de 0.59% es invariante respecto al modelo:

```
n_eventos / n_filas = 140 / 23,800 ≈ 0.59%  — independiente del modelo
```

CoxFrailty añade un parámetro adicional (θ) que requiere aún más eventos
para identificarse correctamente. Con datos suficientes, la frailty capturaría
heterogeneidad real entre motores y mejoraría las predicciones. Con 0.4%
evento rate, θ no es identificable y el modelo colapsa a CoxPH estándar
con estabilidad numérica mejorada.

**Ranking de modelos de supervivencia**

| Métrica | CoxPH | **CoxFrailty** | AFT |
|---------|-------|----------------|-----|
| S-Score | 7,182 | 3,390 | **210** |
| MAE | 32.5 | **29.4** | 38.5 |
| RMSE | 49.4 | **45.5** | 47.2 |
| C-Index | 0.544 | 0.569 | **0.855** |

CoxFrailty es intermedio en S-Score/MAE/RMSE pero su C-Index es similar
a CoxPH — muy por debajo de AFT. AFT sigue siendo el modelo de supervivencia
con mayor capacidad discriminativa real.

**Comparación acumulada completa**

| Métrica | NB | DT | RF | SVR | XGB | CoxPH | AFT | **Frailty** |
|---------|----|----|-----|-----|-----|-------|-----|-------------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | 1.782 | 7,182 | 210 | **3,390** |
| MAE | 10.39 | 7.79 | 6.98 | 6.75 | 7.10 | 32.5 | 38.5 | **29.4** |
| RMSE | 13.36 | 12.22 | 10.62 | 10.24 | 10.45 | 49.4 | 47.2 | **45.5** |
| C-Index | 0.908 | 0.893 | 0.909 | 0.915 | 0.912 | 0.544 | 0.855 | **0.569** |

En este sentido, CoxFrailty aporta un resultado metodológico valioso: demuestra que el término
de frailty estabiliza numéricamente la estimación Cox con datos escasos (100%
convergencia vs 22.2% de CoxPH), pero no resuelve la incompatibilidad
estadística con el 0.4% evento rate. El método de estimación de θ (em vs aic)
es completamente irrelevante — confirmando que θ no es identificable con
los datos disponibles.

**No se seleccionan hiperparámetros de producción.** CoxFrailty queda
documentado como resultado negativo con valor metodológico doble: confirma
la incompatibilidad estructural de los modelos de supervivencia con el
pipeline de ventanas deslizantes en C-MAPSS FD001, y demuestra que añadir
términos de heterogeneidad aleatoria no resuelve el problema cuando la
causa raíz es la escasez de eventos — no la violación del supuesto de
independencia.